In [2]:
# ========== 导入：LangChain RAG 组件 + Gradio UI ==========
# 练习目标（第 5 周 Day5）：SemanticChunker 分块 + Chroma 向量库 + 会话式检索链

# os：读环境变量（Environment Variables）
import os
# load_dotenv：从 .env 加载密钥，避免把 API Key 写进代码
from dotenv import load_dotenv
# ChatOpenAI：聊天模型封装（后面温度设 0 求稳定回答）
from langchain.chat_models import ChatOpenAI
# ConversationalRetrievalChain：带对话记忆的检索增强生成链
from langchain.chains import ConversationalRetrievalChain
# ConversationBufferMemory：把多轮 chat_history 存下来
from langchain.memory import ConversationBufferMemory
# Chroma：向量存储（Vector Store）
from langchain.vectorstores import Chroma
# OpenAIEmbeddings：调用 OpenAI 嵌入模型
from langchain.embeddings import OpenAIEmbeddings
# DirectoryLoader / TextLoader：目录批量加载文本（本格未直接用，保留导入）
from langchain.document_loaders import DirectoryLoader, TextLoader
# RecursiveCharacterTextSplitter：字符递归分块（下方被注释，备选）
from langchain.text_splitter import RecursiveCharacterTextSplitter
# SemanticChunker：按语义边界切块（本练习实际启用）
from langchain_experimental.text_splitter import SemanticChunker
# Document：LangChain 文档对象（page_content + metadata）
from langchain.schema import Document
# gradio：快速搭 Web 聊天界面
import gradio as gr
# glob：按模式枚举 knowledge_base 下的 .md 路径
import glob


In [3]:
# ========== 加载环境变量并读取 OpenAI API Key ==========
# override=True：已有同名环境变量时也以 .env 覆盖
load_dotenv(override=True)
# 后续 OpenAIEmbeddings / ChatOpenAI 会用到该密钥
api_key = os.getenv('OPENAI_API_KEY')


In [ ]:
# ========== 从 knowledge_base 递归加载全部 Markdown ==========
# 路径模式：任意子目录下的 .md；需在含 knowledge_base 的工作目录运行
file_paths = glob.glob("knowledge_base/**/*.md", recursive=True)

documents = []
for path in file_paths:
    # utf-8 读全文，避免中文/特殊字符乱码
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
        # 用父目录名当 doc_type（如 employees / products）
        doc_type = os.path.basename(os.path.dirname(path)) 

        # 包装成 LangChain Document，供后续 SemanticChunker 使用
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "doc_type": doc_type,
                },
            )
        )


In [ ]:
# ========== 嵌入模型 + 语义分块（Semantic Chunking）==========
# text-embedding-3-small：成本较低的 OpenAI 嵌入；密钥用上一格的 api_key
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=api_key)

# 备选：固定长度递归切分（当前注释掉，逻辑保持原样）
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
# SemanticChunker：用嵌入相似度找语义边界，而不是死按字符数切
text_splitter = SemanticChunker(embeddings)
# 把整篇 Document 切成多个较小 chunk
chunks = text_splitter.split_documents(documents)

print(f"Total number of chunks: {len(chunks)}")


In [ ]:
# ========== 用分块结果构建并持久化 Chroma 向量库 ==========
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    # 落盘目录名；下次可复用同路径加载
    persist_directory="chroma_db"
)
# 显式 persist，确保写入磁盘
vectorstore.persist()
print("Chroma vector store built.")


In [ ]:
# ========== 组装会话式 RAG：LLM + Memory + Retriever + Chain ==========
# gpt-4o-mini + temperature=0：便宜且回答更稳定
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=api_key)
# memory_key 必须与链期望的 chat_history 字段名一致
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
# 默认检索器：从 vectorstore 取相关块
retriever = vectorstore.as_retriever()
# ConversationalRetrievalChain：先检索再带着历史问 LLM
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
)


In [ ]:
# ========== 冒烟测试：对会话链提一个固定问题 ==========
# 问题字符串保持英文原样（影响检索与回答）
query = "Tell me about Langchain."
# 旧式调用：传入 {"question": ...}；链内部会写 memory
result = conversation_chain({"question": query})

print("Answer:")
# 答案在 result["answer"]
print(result["answer"])


In [ ]:
# ========== Gradio 聊天 UI：把 conversation_chain 包成可点开的助手 ==========

def rag_chat(query, history):
    # Gradio 会传入 history，但本实现仍依赖链内 ConversationBufferMemory
    response = conversation_chain({"question": query})
    answer = response["answer"]
    return answer

# Soft 主题的 Blocks 容器
with gr.Blocks(theme=gr.themes.Soft()) as rag_ui:
    # 界面展示文案（给人看的 Markdown）；保持原英文 UI 字符串，避免改产品文案行为
    gr.Markdown("# RAG Chat Assistant")
    gr.Markdown("Ask questions about your Markdown knowledge base.")
    # ChatInterface：把 rag_chat 接到对话框
    chat_box = gr.ChatInterface(
        fn=rag_chat,
        title="RAG Knowledge Base Assistant",
        description="Chat with your Markdown-based knowledge base using RAG."
    )


In [ ]:
# ========== 启动 Gradio：debug 看报错，share=True 生成临时公网链接 ==========
rag_ui.launch(debug=True, share=True)
